# SupplementRx AI — Schema Design and Mapping

**Status: Step 1 question/field coverage review documented below. Steps 2–7 have not started; the shared schema remains provisional.**

This notebook will turn the audit findings and provisional POC field selections into a shared schema and mapping rules. Refer to `Data_Audit_and_Preprocessing.ipynb` for the source audits, examples and selected fields. The shared schema is not yet finalized.

## Broad steps planned

1. **Review concrete POC questions and selected fields.** Define representative questions, identify the data needed to answer them, and document coverage gaps and evidence limits.
2. **Review cross-source identities and terminology mappings.** Examine PubChem, RxNorm and MeSH together, linked to originating DSLD ingredient/form occurrences. Distinguish equivalent identities, specific forms, broader concepts and unresolved candidates while preserving separate source identifiers.
3. **Draft the shared conceptual schema.** Define entities and relationships for product labels, ingredients, terminology concepts, NatMed clinical-reference assertions, Reddit user reports and source evidence. Keep evidence roles and source scope explicit.
4. **Define source-to-schema field mappings.** Specify how selected fields map to the shared model, including field types, missing-value states, source context, timestamps and provenance.
5. **Specify identity, normalization and deduplication rules.** Document safe representation changes, candidate-review requirements, assertion-equivalence criteria, conflict handling and update/version policies. Rule design comes here; applying transformations comes later.
6. **Review the draft against representative examples.** Check that it can represent equivalent units, distinct ingredient forms, different serving bases, source conflicts, clinical context and inherited Reddit topics without losing information. Document open decisions.
7. **Finalize the design and prepare the implementation handoff.** Consolidate the agreed schema, mapping rules and acceptance checks. Plan a small normalization pilot, followed by validation, deduplication and eventual graph construction in a separate implementation notebook.

## Working boundaries

- Keep raw datasets unchanged and retain original values alongside any future derived representations.
- Treat PubChem, RxNorm and MeSH as a coordinated terminology layer with distinct identities and reviewable mappings; a shared query or name does not establish equivalence.
- Keep DSLD label information, NatMed clinical-reference assertions and Reddit experiences distinguishable. Source traceability and structural validity do not establish clinical correctness.
- Preserve unresolved identities, missing context and conflicting assertions rather than guessing corrections or forcing merges.
- Consider stable identifiers, explicit relationship types and provenance for later data additions, graph storage and possible GNN/RGCN work; do not implement those stages here.
- Do not update `read.md` with the design until the shared schema is finalized.

**Current scope:** review what the selected fields can support. No cross-source identity decisions, normalization, deduplication or graph construction have been performed. The next planned step is the combined terminology review, after reviewing the Step 1 findings.


## Step 1 — Review concrete POC questions and selected fields

**Deliverable:** a proposed question set, source/field coverage matrix and acceptance criteria. These are evaluation scenarios, not generated medical answers or approved user requirements. The local audits and selected-field cells in `Data_Audit_and_Preprocessing.ipynb` are the reference. No new data collection or transformations are performed.

The questions cover the five user needs: purpose/effectiveness, safety/interactions, amount/usage context, product composition/comparison and evidence transparency. Evidence transparency applies to every question, not only Q10.

Coverage labels below mean:

- **Direct source retrieval:** the selected fields support an attributed answer about a saved source. It does not establish present-day product availability or clinical truth.
- **Rule-dependent:** needed fields exist, but answering comparably requires later normalization/context rules.
- **Review-dependent:** candidate evidence is present, but identity, clinical scope or source interpretation remains unresolved. The POC may show qualified source excerpts, not silently treat pending annotations as approved facts.
- **Unresolved:** the answer should expose the limitation instead of inventing a value or identity.

These labels describe the feasibility of a future pipeline, not a functioning retrieval system.


### Question-to-field coverage matrix

| ID / user need | Concrete POC question | Selected fields and sources | Current coverage / boundary |
|---|---|---|---|
| **Q1 — Amount / product composition** | “How much magnesium does the saved Jarrow Magnesium Optimizer label list per serving?” | DSLD `id`, `brandName`, `fullName`; ingredient ID/name/forms; amount/unit/operator and serving basis; `servingSizes`; source/market context | **Direct source retrieval.** Labels 7841 and 212304 list 100 mg against different tablet counts. Present the selected label/version explicitly; do not return an unqualified shared dose. |
| **Q2 — Product comparison** | “Do BPI Creatine Alkaline labels 25731 and 43261 list the same creatine amount per two capsules?” | DSLD quantity/unit/operator/basis and ingredient forms; label provenance; future normalization-rule metadata | **Rule-dependent.** 1500 mg and 1.5 Gram(s) are equivalent mass representations at the same stated basis. This does not establish that the entire labels or formulations are duplicates. |
| **Q3 — Composition / label precautions** | “Which other ingredients and precautions appear on each of those BPI labels?” | DSLD `otheringredients.text`, ingredient names/IDs/forms; selected `statements.type`/`notes`; source label ID | **Direct source retrieval.** The labels differ in other ingredients. Keep label-specific lists and warnings. Missing text does not prove absence of an ingredient, allergy risk or precaution. |
| **Q4 — Purpose / effectiveness** | “What uses and effectiveness ratings does the saved NatMed Melatonin monograph describe?” | NatMed `ingredient`, `claim_type=effectiveness`, `indication`, `effectiveness_rating`, `population`, `claim_text`, `source`, `field_support`, `review`; scoped references | **Review-dependent.** 104 effectiveness annotations have indications, but all 104 lack structured population values. Retrieve the original qualification text; do not generalize across populations or promote pending claims into recommendations. |
| **Q5 — Safety context** | “What safety statements and population qualifications does the saved Ashwagandha monograph contain?” | NatMed safety annotations: ingredient/formulation, population, safety rating, full excerpt and source spans; original warning/overview sections when needed; review flags | **Review-dependent.** Seven safety annotations exist; six have structured populations. An upstream extraction gap affects overview-embedded warnings, so claims alone cannot support a complete safety summary. |
| **Q6 — Interactions** | “Which interaction partners and qualifications does the Magnesium monograph report?” | NatMed `claim_type=interaction`, partner/rating/severity/occurrence, evidence grade, population, `claim_text`; source section heading, field supports and review state | **Review-dependent.** 24 interaction annotations exist. Use section context to distinguish partner types; absence from the extracted set does not mean no interaction. No personalized compatibility decision is supported. |
| **Q7 — Dose context** | “What dosing contexts and regimens are described in the saved Creatine monograph?” | NatMed full `dose`, ingredient/formulation, population, indication, original excerpt, source support and review status | **Review-dependent.** Two dose annotations exist with populations but no structured indications. Preserve regimen, bounds, frequency and source text. Do not describe all values as study-specific doses or recommendations without source support. |
| **Q8 — Ingredient identity / terminology** | “Does the DSLD name Creatine HCl have an equivalent entry in PubChem, RxNorm and MeSH?” | Original DSLD occurrence; PubChem CID/properties and name-query candidates; RxNorm RxCUI/name/TTY and search method; MeSH term–concept–record links; provenance and candidate review status | **Unresolved identity.** PubChem returned CID 134732; RxNorm normalized search returned creatine and creatine monohydrate; MeSH returned no candidate. Show the disagreement; do not join all results as one ingredient. Actual mapping review is Step 2. |
| **Q9 — User questions / experiences** | “What are people in the saved Reddit sample asking or reporting about melatonin?” | Reddit title/text, source URL, record/native ID, subreddit, dates, `supplements.matched_terms`, `mention_scope`, thread/parent IDs, `evidence_role`; passage bindings | **Direct retrieval with relevance review.** There are 759 topic-associated records: 135 `own_text`, 624 `thread_context`. Scope is record-level: even `own_text` does not guarantee melatonin is the particular directly matched topic. Verify topic relevance in the actual text; qualify anecdotes and missing context. |
| **Q10 — Provenance / conflicting values** | “Why do the BetterYou Iron Spray records show different serving volumes, and which original fields support those values?” | DSLD label IDs 228638/257287, original quantity/serving fields and notes, source paths/hashes, conflict flags; original evidence | **Conflict explainable; correction unresolved.** Label 257287 has numeric 52 mL versus 0.52 ml in its own note. Show the source discrepancy and flag the affected calculation. Do not claim a verified cause or silently choose the corrected value. |

Every future answer needs source-specific evidence links, applicable label/version or monograph context, and any material unresolved status. A concise answer may hide internal identifiers by default, but they must remain traceable.


### Coverage findings and selected-field review

**1. Product-label questions have the strongest direct coverage.** DSLD supports composition, stated quantities, directions and precautions for saved labels. Preserve other ingredients: Q2 can have equivalent active-ingredient quantities while Q3 differs. Product identity, package/formulation identity and assertion equivalence must remain separate decisions.

**2. NatMed supports contextual source retrieval, not automatic clinical approval.** Its selected fields cover purpose, safety, interactions and dose context. Unknown structured population/formulation/indication fields and extraction gaps make the full excerpt and source section essential. All annotations remain pending despite passing structural/source-span checks. A future answer should attribute what the reference states and expose relevant limits.

**3. The terminology sources must be reviewed together.** Q8 requires a side-by-side candidate view anchored to the same DSLD name/form occurrence. PubChem, RxNorm and MeSH have different coverage and concept scope; missing matches must not discard the source ingredient. Shared IDs, names, formulas or synonyms cannot replace mapping review.

**4. Reddit requires conversation context and a distinct evidence role.** Its selected fields support finding relevant user questions and experiences. They do not yet support structured efficacy, dose or interaction conclusions. Inherited topics, repeated text, passage overlap and context gaps must not inflate apparent independent evidence or imply direct use.

**5. Citations and source context cannot be optional internal fields.** Every source must retain identity, source URL/path, snapshot/hash and extraction context. NatMed reference linkage needs claim-specific support; all monograph references cannot be attached to every claim. Reddit URLs attribute a user report, while terminology API URLs attribute a vocabulary result—not clinical evidence.

**Field-selection outcome:** retain the current provisional selections. No additional clinical fields are invented, and no source fields are removed during this step. In particular, keep:

- ingredient forms, nested/blend context, amount operator and serving basis;
- other ingredients and original directions/precautions;
- full clinical excerpts, source section, population/indication and dose qualifiers even when unknown;
- topic mention scope and thread/parent references;
- mapping method, complete candidate sets, provenance, source-specific dates and review/conflict flags.

**Gaps that need later work, not guessed values:** accepted cross-source ingredient mappings; source-specific product/version identity; incomplete dose/population context; claim-specific citation support; source conflicts; and Reddit topic relevance. Detailed schema requirements and entity relationships are intentionally deferred to Steps 2–4.

No source supports a blanket answer such as “this product is currently available,” “this supplement is safe for you,” or “this dose is appropriate for your medications” merely because an ingredient name matches. These are outside the initial question set.


### Acceptance criteria for the later POC implementation

| Scenario | Expected behavior | Failure to prevent |
|---|---|---|
| Label amount lookup (Q1) | State the amount and exact serving basis with label/version context | Returning 100 mg without specifying one versus two tablets |
| Equivalent units (Q2) | Compare compatible mass values while preserving originals and subject context | Declaring complete products identical from a single matching quantity |
| Other ingredients / precautions (Q3) | Attribute differences to the correct label | Losing composition differences during deduplication or treating missing text as no risk |
| Clinical-reference retrieval (Q4–Q7) | Retain source text, qualifications, review status and scope | Inferring missing population/indication, converting a bound into an exact recommendation, or treating pending annotations as approved |
| Terminology disagreement (Q8) | Display source-specific candidates and unresolved identity | Treating creatine, monohydrate and hydrochloride as interchangeable |
| Reddit retrieval (Q9) | Verify the queried topic in context and label statements as user reports | Presenting inherited thread topics as direct experiences or clinical evidence |
| Source conflict (Q10) | Expose the conflicting fields, retain evidence and flag derived calculations | Silently correcting 52 mL or averaging conflicting values |
| Evidence across all questions | Trace each factual statement to its source/context | Counting duplicate labels or overlapping passages as independent efficacy evidence |

These are design acceptance criteria, **not executed end-to-end tests**. The counts used in this review were checked against the current local source files; no generated medical answers or clinical assessments were produced.

**Step 1 outcome:** a concrete, proposed question set and source/field coverage review are documented. The next planned activity is **Step 2: combined terminology and cross-source identity review**, using Q8 and the label examples to guide scope. No Step 2 work, normalization, merging or graph construction has been performed here. `read.md` remains unchanged.
